# Fábrica dos Cortes — Colab

Transforme vídeos longos em cortes virais direto no Google Colab, com GPU.

**Antes de começar**

1. No menu **Ambiente de execução → Alterar o tipo de ambiente de execução**, escolha **T4 GPU** (ou outra GPU).
2. Rode a célula **1** e espere a instalação terminar (10 a 15 minutos).
3. Rode a célula **2** se quiser guardar seus projetos no Google Drive (recomendado).
4. Rode a célula **3**, espere aparecer o link público `https://....gradio.live` e abra.

**Versões travadas** (as mesmas do instalador oficial 1.3.6):

- Python 3.10 isolado
- PyTorch 2.5.1 (CUDA 12.1 ou CPU), torchvision 0.20.1, torchaudio 2.5.1
- WhisperX 3.4.5, Transformers 4.48.0, CTranslate2 4.8.4, Pyannote Audio 3.4.0
- Gradio 6.6.0, NumPy 2.0.2

Tudo é instalado num ambiente virtual dentro de `/content/forjaviral`, sem mexer no Python do Colab.

In [ ]:
#@title 1. Instalar o app e as dependências (10 a 15 min)
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path("/content/forjaviral")

if ROOT.exists():
    print("Removendo instalação anterior...")
    shutil.rmtree(ROOT)

print("Clonando o repositório oficial...")
subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/bazotepedro-oss/forjaviral.git", str(ROOT)],
    check=True,
)

print("Instalando FFmpeg, Aria2 e bibliotecas do sistema...")
subprocess.run(
    "apt-get -qq update && apt-get -qq install -y ffmpeg aria2 libgl1 libglib2.0-0 libsndfile1 > /dev/null",
    shell=True,
    check=True,
)

print("Instalando o gerenciador uv...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

venv_python = str(ROOT / ".venv" / "bin" / "python")
print("Criando Python 3.10 isolado...")
subprocess.run(["uv", "venv", "--python", "3.10", ".venv"], cwd=ROOT, check=True)

has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
torch_index = "https://download.pytorch.org/whl/cu121" if has_gpu else "https://download.pytorch.org/whl/cpu"
print("GPU detectada:", has_gpu)

print("Instalando PyTorch 2.5.1 (pode demorar)...")
subprocess.run(
    [
        "uv", "pip", "install", "--python", venv_python,
        "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
        "--index-url", torch_index,
    ],
    cwd=ROOT,
    check=True,
)

print("Instalando as dependências travadas do app...")
subprocess.run(
    [
        "uv", "pip", "install", "--python", venv_python,
        "-r", "requirements.txt", "-c", "constraints.txt",
        "--extra-index-url", torch_index,
    ],
    cwd=ROOT,
    check=True,
)

# O WhisperX pede uma versão antiga do CTranslate2; a 4.8.4 funciona com o
# cuDNN que o PyTorch instala. Instalamos sem dependências para não conflitar.
print("Ajustando o CTranslate2 para 4.8.4...")
subprocess.run(
    ["uv", "pip", "install", "--python", venv_python, "--no-deps", "ctranslate2==4.8.4"],
    cwd=ROOT,
    check=True,
)

print("Conferindo as versões instaladas...")
subprocess.run(
    [
        venv_python,
        "-c",
        (
            "import gradio, torch, transformers, ctranslate2, whisperx; "
            "print('gradio', gradio.__version__); "
            "print('torch', torch.__version__, '| cuda', torch.cuda.is_available()); "
            "print('transformers', transformers.__version__); "
            "print('ctranslate2', ctranslate2.__version__); "
            "print('whisperx ok')"
        ),
    ],
    cwd=ROOT,
    check=True,
)

print("\n✅ Instalação concluída. Pode rodar a célula 2 ou 3.")

## 2. Guardar os projetos no Google Drive (opcional, recomendado)

Sem isso, os vídeos e cortes somem quando o Colab for encerrado.
A pasta `VIRALS` do app passa a apontar para `Meu Drive/FabricaDosCortes/VIRALS`.

In [ ]:
#@title 2. Montar o Google Drive e ligar a pasta VIRALS
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

project_root = Path("/content/forjaviral")
drive_dir = Path("/content/drive/MyDrive/FabricaDosCortes/VIRALS")
drive_dir.mkdir(parents=True, exist_ok=True)

local_virals = project_root / "VIRALS"
if local_virals.is_symlink():
    local_virals.unlink()
elif local_virals.exists():
    # se já houver algo local, move para o Drive antes de criar o link
    for item in local_virals.iterdir():
        target = drive_dir / item.name
        if not target.exists():
            item.rename(target)
    local_virals.rmdir()

local_virals.symlink_to(drive_dir, target_is_directory=True)
print("VIRALS agora aponta para:", drive_dir)

## 3. Iniciar a Fábrica dos Cortes

Aguarde o link público. Ele é temporário e cai quando o Colab encerrar.

**Chave Gemini:** você pode colar na aba **Criar Cortes** do app. Se preferir,
cadastre o segredo `GEMINI_API_KEY` nos **Segredos do Colab** (ícone de chave na
barra lateral) que a célula preenche sozinha.

**Licença:** o app pede a chave de licença na aba **Licença**, como no instalador.

In [ ]:
#@title 3. Iniciar o app e gerar o link público
import json
import os
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["PYTHONUNBUFFERED"] = "1"

root = Path("/content/forjaviral")
config_path = root / "api_config.json"

# Chave Gemini via Segredos do Colab (opcional)
try:
    from google.colab import userdata

    gemini_key = (userdata.get("GEMINI_API_KEY") or "").strip()
    if gemini_key:
        config = json.loads(config_path.read_text(encoding="utf-8-sig"))
        config.setdefault("gemini", {})["api_key"] = gemini_key
        config_path.write_text(json.dumps(config, indent=4), encoding="utf-8")
        print("Chave Gemini carregada dos Segredos do Colab.")
    else:
        print("Segredo GEMINI_API_KEY vazio: cole a chave na interface do app.")
except Exception:
    print("Sem Segredos do Colab: cole a chave Gemini na interface do app.")

print("Iniciando... aguarde o link público (leva de 30 a 60 segundos).")
print("Se pedir confirmação para compartilhar, aceite.")
!cd /content/forjaviral && .venv/bin/python webui/app.py --colab

## Como usar

1. Abra o link público que apareceu acima.
2. Vá em **Licença** e cole sua chave.
3. Volte em **Criar Cortes**, cole o link do YouTube e escolha quantos cortes quer.
4. Clique em **Gerar Cortes** e acompanhe a barra de progresso.

Os cortes prontos aparecem na galeria e ficam em `VIRALS/<nome do vídeo>`.
Com o Drive montado, eles continuam lá na próxima sessão.

**Problemas comuns**

- **GPU indisponível**: confira em *Ambiente de execução → Alterar o tipo de ambiente de execução* se está em T4 GPU.
- **Link não aparece**: rode a célula 3 de novo; o Gradio às vezes demora.
- **Cota do Gemini (429)**: o app troca de modelo sozinho; se todos esgotarem, espere o reset da cota.
- **Sessão caiu**: no Colab gratuito a sessão dura algumas horas. Rode as células 1, 2 e 3 novamente (a 1 reinstala; se quiser pular, clone e venv continuam no disco enquanto a sessão viver).